In [1]:
# ============================================================
# FEATURE EXTRACTION PIPELINE — all 4 frozen models × eval datasets
# Reads committed model + unified tables from mounted notebook output.
# Filters missing-image rows. Resumable per (model, dataset).
# ============================================================
!pip install --quiet open_clip_torch

import os, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# --- HF auth (for DINOv3, gated) ---
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(token=UserSecretsClient().get_secret("HF_TOKEN"))

# --- Paths ---
SRC = "/kaggle/input/notebooks/nirajankunwor/skin-tone-classification"
UNIFIED = f"{SRC}/unified"
BASELINE_MODEL = f"{SRC}/baseline_resnet50_best.pt"
OUT = "/kaggle/working/embeddings"
os.makedirs(OUT, exist_ok=True)

# ============================================================
# LOAD EVAL DATASETS — with missing-image filtering
# ============================================================
def load_and_filter(path, name):
    df = pd.read_csv(path)
    n0 = len(df)
    df = df[df["image_path"].apply(os.path.exists)].reset_index(drop=True)
    dropped = n0 - len(df)
    if dropped:
        print(f"{name}: dropped {dropped} missing-image rows ({dropped/n0*100:.2f}%), {len(df)} remain")
    else:
        print(f"{name}: all {len(df)} images present")
    return df

eval_sets = {
    "ddi":  load_and_filter(f"{UNIFIED}/ddi_unified.csv", "ddi"),
    "scin": load_and_filter(f"{UNIFIED}/scin_unified.csv", "scin"),
    "hamisic_test": load_and_filter(f"{UNIFIED}/baseline_test.csv", "hamisic_test"),
}
ORDER = ["ddi", "scin", "hamisic_test"]

# ============================================================
# GENERIC EXTRACTION MACHINERY
# ============================================================
class ImgDataset(Dataset):
    def __init__(self, df, preprocess):
        self.paths = df["image_path"].tolist()
        self.preprocess = preprocess
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.preprocess(img), i

def extract(model_forward, preprocess, df, batch_size=64):
    loader = DataLoader(ImgDataset(df, preprocess), batch_size=batch_size,
                        shuffle=False, num_workers=2, pin_memory=True)
    embs = [None] * len(df)
    with torch.inference_mode():
        for batch, idxs in loader:
            batch = batch.to(device)
            out = model_forward(batch).float().cpu().numpy()
            for j, idx in enumerate(idxs.numpy()):
                embs[idx] = out[j]
    return np.stack(embs)

IMAGENET_TF = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
CLIP_TF = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.48145466, 0.4578275, 0.40821073],
                         [0.26862954, 0.26130258, 0.27577711]),
])

# ============================================================
# MODEL LOADERS
# ============================================================
def load_resnet_baseline():
    m = models.resnet50(weights=None)
    m.fc = nn.Linear(m.fc.in_features, 8)
    m.load_state_dict(torch.load(BASELINE_MODEL, map_location=device))
    m.fc = nn.Identity()
    m = m.to(device).eval()
    return (lambda x: m(x)), IMAGENET_TF

def load_dermlip():
    import open_clip
    m, _, pre = open_clip.create_model_and_transforms('hf-hub:redlessone/DermLIP_ViT-B-16')
    m = m.to(device).eval()
    return (lambda x: m.encode_image(x)), pre

def load_monet():
    from transformers import AutoModel
    m = AutoModel.from_pretrained("chanwkim/monet").to(device).eval()
    return (lambda x: m.vision_model(pixel_values=x).pooler_output), CLIP_TF

def load_dinov3():
    from transformers import AutoModel
    m = AutoModel.from_pretrained("facebook/dinov3-vitb16-pretrain-lvd1689m").to(device).eval()
    return (lambda x: m(pixel_values=x).pooler_output), IMAGENET_TF

MODEL_LOADERS = {
    "resnet_baseline": load_resnet_baseline,
    "dermlip": load_dermlip,
    "monet": load_monet,
    "dinov3": load_dinov3,
}

# ============================================================
# RUN: every model × every dataset, resumable
# ============================================================
for model_name, loader_fn in MODEL_LOADERS.items():
    if all(os.path.exists(f"{OUT}/{model_name}_{ds}.npy") for ds in ORDER):
        print(f"[{model_name}] already complete, skipping.")
        continue
    print(f"\n=== Loading {model_name} ===")
    forward_fn, preprocess = loader_fn()
    for ds_name in ORDER:
        out_path = f"{OUT}/{model_name}_{ds_name}.npy"
        if os.path.exists(out_path):
            print(f"  {ds_name}: done, skip.")
            continue
        df = eval_sets[ds_name]
        print(f"  {ds_name}: extracting {len(df)} images...")
        emb = extract(forward_fn, preprocess, df)
        np.save(out_path, emb)
        meta_path = f"{OUT}/meta_{ds_name}.csv"
        if not os.path.exists(meta_path):
            df.to_csv(meta_path, index=False)
        print(f"    saved {emb.shape} -> {os.path.basename(out_path)}")
    del forward_fn
    gc.collect(); torch.cuda.empty_cache()

print("\n" + "="*50)
print("FEATURE EXTRACTION COMPLETE")
print("="*50)
for f in sorted(os.listdir(OUT)):
    print(" ", f)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.6 MB/s eta 0:00:00
Device: cpu
ddi: all 656 images present
scin: dropped 1 missing-image rows (0.02%), 6517 remain
hamisic_test: all 5378 images present

=== Loading resnet_baseline ===
  ddi: extracting 656 images...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


    saved (656, 2048) -> resnet_baseline_ddi.npy
  scin: extracting 6517 images...
    saved (6517, 2048) -> resnet_baseline_scin.npy
  hamisic_test: extracting 5378 images...
    saved (5378, 2048) -> resnet_baseline_hamisic_test.npy

=== Loading dermlip ===


open_clip_config.json:   0%|          | 0.00/532 [00:00<?, ?B/s]

open_clip_model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

...

  ddi: extracting 656 images...
    saved (656, 1024) -> monet_ddi.npy
  scin: extracting 6517 images...
    saved (6517, 1024) -> monet_scin.npy
  hamisic_test: extracting 5378 images...
    saved (5378, 1024) -> monet_hamisic_test.npy

=== Loading dinov3 ===


config.json:   0%|          | 0.00/744 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/343M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

  ddi: extracting 656 images...
    saved (656, 768) -> dinov3_ddi.npy
  scin: extracting 6517 images...
    saved (6517, 768) -> dinov3_scin.npy
  hamisic_test: extracting 5378 images...
    saved (5378, 768) -> dinov3_hamisic_test.npy

FEATURE EXTRACTION COMPLETE
  dermlip_ddi.npy
  dermlip_hamisic_test.npy
  dermlip_scin.npy
  dinov3_ddi.npy
  dinov3_hamisic_test.npy
  dinov3_scin.npy
  meta_ddi.csv
  meta_hamisic_test.csv
  meta_scin.csv
  monet_ddi.npy
  monet_hamisic_test.npy
  monet_scin.npy
  resnet_baseline_ddi.npy
  resnet_baseline_hamisic_test.npy
  resnet_baseline_scin.npy
